In [1]:
import os, sys

REPO = "/content/tp2-sarcasme-arabe"
if not os.path.exists(REPO):
    !git clone https://github.com/AbdelhamidRb/tp2-sarcasme-arabe.git

os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
!git pull
print(os.listdir("src"))

Cloning into 'tp2-sarcasme-arabe'...
remote: Enumerating objects: 33, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 33 (delta 10), reused 27 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (33/33), 32.74 KiB | 4.09 MiB/s, done.
Resolving deltas: 100% (10/10), done.
Already up to date.
['data.py', 'preprocess.py', '__init__.py', 'seed.py', 'results.py']


In [2]:
import os, sys
REPO = "/content/tp2-sarcasme-arabe"
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

In [3]:
%cd /content/tp2-sarcasme-arabe
!git pull

from src.seed import set_seed
from src.data import load_splits
from src.preprocess import dedupliquer_train

set_seed(42)
train_brut, val, test = load_splits()
train = dedupliquer_train(train_brut, test)
print("taux sarcasme train :", round(train.sarcasm.mean(), 4))

/content
Already up to date.
train avant : 11293
  fuite retirée (présents dans test) : 4
  groupes contradictoires retirés    : 13
train après : 11217
taux sarcasme train : 0.1721


In [4]:
print("Global | train:", round(train.sarcasm.mean(), 4),
      "| val:", round(val.sarcasm.mean(), 4),
      "| test:", round(test.sarcasm.mean(), 4))

for col in ["dialect", "sentiment"]:
    for nom, df in [("train", train), ("test", test)]:
        print(f"\n{col} — {nom}")
        print(df.groupby(col)["sarcasm"].agg(["mean", "count"]).round(4))

Global | train: 0.1721 | val: 0.1729 | test: 0.2737

dialect — train
           mean  count
dialect               
egypt    0.3464   2405
gulf     0.2487    575
levant   0.2120    566
magreb   0.3947     38
msa      0.1074   7633

dialect — test
           mean  count
dialect               
egypt    0.5065    306
gulf     0.1770    322
levant   0.2979     47
magreb   0.0000      2
msa      0.2561   2323

sentiment — train
             mean  count
sentiment               
NEG        0.4191   4116
NEU        0.0300   5136
POS        0.0265   1965

sentiment — test
             mean  count
sentiment               
NEG        0.4228   1677
NEU        0.1417    748
POS        0.0104    575


In [5]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

y_va = val["sarcasm"].astype(int).values
zero = np.zeros_like(y_va)
print("accuracy :", round(accuracy_score(y_va, zero), 4))
print("F1 sarc  :", f1_score(y_va, zero, zero_division=0))

accuracy : 0.8271
F1 sarc  : 0.0


In [6]:
import time
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

vec_mots = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
vec_car = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2)

# fit uniquement sur train (règle 2)
X_tr = hstack([vec_mots.fit_transform(train.tweet), vec_car.fit_transform(train.tweet)]).tocsr()
X_va = hstack([vec_mots.transform(val.tweet), vec_car.transform(val.tweet)]).tocsr()
y_tr = train["sarcasm"].astype(int).values

t0 = time.time()
modele = LogisticRegression(class_weight="balanced", max_iter=2000)
modele.fit(X_tr, y_tr)
t_entrainement = time.time() - t0
print("temps :", round(t_entrainement, 2), "s")

proba_va = modele.predict_proba(X_va)[:, 1]
print("PR-AUC val :", round(average_precision_score(y_va, proba_va), 4))

temps : 4.81 s
PR-AUC val : 0.5217


Exemple fictif : après avoir retenu les 10 premiers tweets, si 7 sont sarcastiques, la précision est 7/10 = 0,70 et le rappel 7/217 ≈ 0,03.

Étape 3 : tracer la courbe. Chaque position donne un point (rappel, précision). L’ensemble forme la courbe précision-rappel.

Étape 4 : mesurer l’aire. La PR-AUC est l’aire sous cette courbe. Elle vaut entre 0 et 1 :

1 = classement parfait (tous les sarcastiques sont en tête) ;
environ 0,17 = classement au hasard (égal à la proportion de sarcastiques).

Notre valeur, 0,5217, signifie que le modèle classe nettement mieux que le hasard, mais qu’il laisse beaucoup de sarcastiques mal placés.

Pourquoi c’est utile : cette mesure ne dépend d’aucun seuil. Elle juge la qualité du classement elle-même, alors que le F1 juge une décision précise.

In [7]:
train.head()

,tweet,sarcasm,sentiment,dialect,id,cle
0,"""#لافروف: #موسكو تأمل بإجراء حوار بناء بشأن ال...",False,NEU,msa,2474,"""#لافروف: #موسكو تأمل بإجراء حوار بناء بشأن ال..."
1,"""RT @jalalyw: @EHSANFAKEEH تفصيل الخبر لمن يري...",False,NEU,msa,12107,"""rt : تفصيل الخبر لمن يريدالإستزادة والإطلاع 1..."
2,تحدثنا عن هذا الكتاب الجميل عن خروج المصريين ع...,False,NEU,msa,2135,تحدثنا عن هذا الكتاب الجميل عن خروج المصريين ع...
3,"""RT @Jokes__99: بوكيمون ولحقتوه عيل المسيح الد...",True,NEG,msa,640,"""rt : بوكيمون ولحقتوه عيل المسيح الدجال عمياني..."
4,"""أوباما يلاحق محمد بن سلمان !!: https://t.co/x...",False,NEU,msa,6766,"""أوباما يلاحق محمد بن سلمان !!: via """


In [8]:
seuils = np.arange(0.05, 0.96, 0.01)
f1s = [f1_score(y_va, proba_va >= t, zero_division=0) for t in seuils]
seuil = seuils[int(np.argmax(f1s))]
print(f"seuil retenu : {seuil:.2f} | F1 sarc val : {max(f1s):.4f}")
print("F1 à 0,5 (référence) :", round(f1_score(y_va, proba_va >= 0.5), 4))

seuil retenu : 0.42 | F1 sarc val : 0.5153
F1 à 0,5 (référence) : 0.5041


In [9]:
import numpy as np
from sklearn.metrics import f1_score

y_va = val["sarcasm"].astype(int).values
baseline_f1 = max(f1s)

signaux = {
    "url": val.tweet.str.contains(r"https?://\S+", regex=True).values,
    "mention": val.tweet.str.contains(r"@\w+", regex=True).values,
    "espace_bord_ou_insecable": val.tweet.map(lambda s: s != s.strip() or "\u00a0" in s).values,
}

for nom, present in signaux.items():
    absent = ~present
    print(f"\n[{nom}] présent dans {present.mean():.3f} des tweets val")
    if present.any():
        print(f"  taux sarcasme si présent : {y_va[present].mean():.4f}")
    if absent.any():
        print(f"  taux sarcasme si absent  : {y_va[absent].mean():.4f}")

    f1_a = f1_score(y_va, present.astype(int), zero_division=0)  # présent => sarcastique
    f1_b = f1_score(y_va, absent.astype(int), zero_division=0)   # absent  => sarcastique
    print(f"  F1 règle 'présent => sarcastique' : {f1_a:.4f}")
    print(f"  F1 règle 'absent  => sarcastique' : {f1_b:.4f}")
    print(f"  Baseline TF-IDF (F1 val)          : {baseline_f1:.4f}")


[url] présent dans 0.398 des tweets val
  taux sarcasme si présent : 0.0822
  taux sarcasme si absent  : 0.2328
  F1 règle 'présent => sarcastique' : 0.1145
  F1 règle 'absent  => sarcastique' : 0.3618
  Baseline TF-IDF (F1 val)          : 0.5153

[mention] présent dans 0.240 des tweets val
  taux sarcasme si présent : 0.2027
  taux sarcasme si absent  : 0.1635
  F1 règle 'présent => sarcastique' : 0.2355
  F1 règle 'absent  => sarcastique' : 0.2664
  Baseline TF-IDF (F1 val)          : 0.5153

[espace_bord_ou_insecable] présent dans 0.055 des tweets val
  taux sarcasme si présent : 0.4058
  taux sarcasme si absent  : 0.1594
  F1 règle 'présent => sarcastique' : 0.1958
  F1 règle 'absent  => sarcastique' : 0.2694
  Baseline TF-IDF (F1 val)          : 0.5153


In [11]:
import sklearn, torch, transformers
from src.results import ajouter_resultat, CHEMIN
import pandas as pd

# Nombre de paramètres entraînés : coefficients + intercept de la régression logistique
n_params = modele.coef_.size + modele.intercept_.size

ajouter_resultat({
    "modele": "TF-IDF + LogisticRegression (class_weight=balanced)",
    "preprocessing": "dedup train + espaces normalisés; mots(1-2) + char_wb(2-5), min_df=2",
    "graine": 42,
    "seuil": round(float(seuil), 2),
    "pr_auc_val": round(float(average_precision_score(y_va, proba_va)), 4),
    "f1_sarc_val": round(float(max(f1s)), 4),
    "f1_sarc_val_seuil_0_5": round(float(f1_score(y_va, proba_va >= 0.5)), 4),
    "accuracy_val_seuil": round(float(accuracy_score(y_va, proba_va >= seuil)), 4),
    "n_train": len(train),
    "n_val": len(val),
    "temps_entrainement_s": round(float(t_entrainement), 2),
    "params_entraines": int(n_params),
    "versions": f"sklearn={sklearn.__version__}; torch={torch.__version__}; transformers={transformers.__version__}",
    "note": "seuil et F1 choisis sur val (optimiste) ; test non utilisé",
})

print(pd.read_csv(CHEMIN).T)

                                                                       0  \
modele                 TF-IDF + LogisticRegression (class_weight=bala...   
preprocessing          dedup train + espaces normalisés; mots(1-2) + ...   
graine                                                                42   
seuil                                                               0.42   
pr_auc_val                                                        0.5217   
f1_sarc_val                                                       0.5153   
f1_sarc_val_seuil_0_5                                             0.5041   
accuracy_val_seuil                                                0.7721   
n_train                                                            11217   
n_val                                                               1255   
temps_entrainement_s                                                3.39   
params_entraines                                                  186332   
versions    

SecretNotFoundError: Secret TP_sarcasme does not exist.